# DroneAI Stage 2 — CSRNet pipeline smoke

This notebook validates the paper-compatible CSRNet architecture, count-preserving density maps, deterministic initialization and tiny-set learning. It uses four synthetic samples and does not claim paper reproduction or real-data accuracy.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/DroneAI')
OUTPUT_DIR = DRIVE_ROOT / 'runs' / 'stage-2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

In [ ]:
import os, shutil, stat, subprocess
from google.colab import userdata
REPO_URL = 'https://github.com/LuciTa81/DroneAI.git'
REPO_DIR = Path('/content/DroneAI')
REPO_REF = 'agent/stage2-csrnet-smoke'
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add the GITHUB_TOKEN secret and grant notebook access.')
askpass = Path('/tmp/droneai_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'password' in prompt else 'x-access-token')\n", encoding='utf-8')
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
try:
    if REPO_DIR.exists() and not (REPO_DIR / '.git').is_dir():
        shutil.rmtree(REPO_DIR)
    if (REPO_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO_DIR), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_REF], env=git_env, check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_REF, f'origin/{REPO_REF}'], check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, str(REPO_DIR)], env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    del token
    git_env.pop('GITHUB_TOKEN', None)
subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

In [ ]:
import sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running Stage 2.')
print(torch.__version__, torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

In [ ]:
result = subprocess.run([
    sys.executable, 'scripts/run_stage2.py',
    '--output-dir', str(OUTPUT_DIR),
    '--steps', '200',
    '--learning-rate', '0.0001',
    '--device', 'cuda',
], cwd=REPO_DIR, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('Exit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('Stage 2 gate did not pass; inspect failed checks.')

In [ ]:
import json, pandas as pd
from IPython.display import Markdown, display
score = json.loads((OUTPUT_DIR / 'score.json').read_text(encoding='utf-8'))
metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
display(Markdown((OUTPUT_DIR / 'score.md').read_text(encoding='utf-8')))
display(pd.DataFrame(score['checks'])[['category', 'description', 'weight', 'earned', 'blocker', 'observed']])
print({key: metrics[key] for key in ('initial_loss', 'final_loss', 'loss_reduction', 'target_counts', 'predicted_counts', 'count_mae')})